# 03 - Natural Language Processing & Vectorization Pipeline
## Customer Feedback Intelligence System

### Objectives:
- Clean and normalize customer review text using Regular Expressions (Regex).
- Perform tokenization, stopword filtering, and lemmatization via NLTK.
- Analyze n-gram frequencies (unigrams & bigrams) for negative vs. positive feedback.
- Train and persist `TfidfVectorizer` for downstream classification and clustering.

In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Ensure NLTK corpora are downloaded
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

DATA_PATH = Path("../data/processed/cleaned_reviews.csv")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

In [2]:
# 1. Text Preprocessing Pipeline Setup
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

# Add domain-specific noise words that carry no discriminative sentiment value
custom_stops = {"dress", "shirt", "top", "pants", "retailer", "bought", "ordered", "look", "wear"}
stop_words = stop_words.union(custom_stops)

def clean_and_normalize(text: str) -> str:
    # Lowercase
    text = str(text).lower()
    # Remove URLs, mentions, and special HTML characters
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"&\w+;", " ", text)
    # Remove non-alphabet characters
    text = re.sub(r"[^a-z\s]", " ", text)
    # Tokenize and remove extra whitespace
    tokens = text.split()
    # Lemmatize and filter short tokens and stopwords
    clean_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return " ".join(clean_tokens)

# Apply cleaning function to entire dataset
df["Processed_Review"] = df["Review Text"].apply(clean_and_normalize)
df[["Review Text", "Processed_Review"]].head(5)

,Review Text,Processed_Review
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable
1,Love this dress! it's sooo pretty. i happene...,love sooo pretty happened find store glad neve...
2,I had such high hopes for this dress and reall...,high hope really wanted work initially petite ...
3,"I love, love, love this jumpsuit. it's fun, fl...",love love love jumpsuit fun flirty fabulous ev...
4,This shirt is very flattering to all due to th...,flattering due adjustable front tie perfect le...


In [3]:
# 2. Extract Top N-grams for Qualitative Analysis
from collections import Counter

def get_top_ngrams(corpus, n=1, top_k=15):
    vec = TfidfVectorizer(ngram_range=(n, n), stop_words=list(stop_words)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq[:top_k], columns=["ngram", "tfidf_sum"])

# Inspect frequent terms in Negative Reviews (Complaints)
neg_reviews = df[df["Rating"] <= 2]["Processed_Review"]
top_neg_unigrams = get_top_ngrams(neg_reviews, n=1, top_k=10)
print("Top Complaint Unigrams (TF-IDF Weight):")
print(top_neg_unigrams)

Top Complaint Unigrams (TF-IDF Weight):
      ngram  tfidf_sum
0      like  87.945932
1      size  74.310352
2       fit  72.154615
3    fabric  70.601520
4     would  67.555606
5      back  63.757644
6     small  58.508372
7     color  57.182328
8  material  53.345412
9    really  53.188079


In [4]:
# 3. Fit Production TF-IDF Vectorizer
# ngram_range=(1, 2) captures phrases such as "poor quality", "runs small"
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    min_df=3,
    max_df=0.85
)

X_tfidf = vectorizer.fit_transform(df["Processed_Review"])
print("TF-IDF Matrix Shape:", X_tfidf.shape)

# Save vectorizer to disk
joblib.dump(vectorizer, MODELS_DIR / "tfidf_vectorizer.pkl")
print("Vectorizer successfully serialized to models/tfidf_vectorizer.pkl")

TF-IDF Matrix Shape: (22626, 5000)
Vectorizer successfully serialized to models/tfidf_vectorizer.pkl
